In [1]:
import numpy as np
import torch
import torchvision.models
import torch.nn as nn
import pandas as pd
import time
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent.parent.parent))
from models.utils import set_seed, train, evaluate_model, plot_training_history, get_loaders
from models.baseline_cnn import BaselineCNN
from models.alexnet import ModifiedAlexNet

sys.path.append(str(Path().resolve().parent))
from hyperparameter_plots import summarize_results, print_summary_table, plot_loss_comparison, plot_accuracy_comparison
sys.path.append(str(Path().resolve().parent.parent))
from utils import save_results

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print("Using device:", device)

Using device: mps


In [2]:
DATA_DIR = Path("../../../data")

TRAIN_DIR = DATA_DIR / "train"
VALID_DIR = DATA_DIR / "valid"
TEST_DIR = DATA_DIR / "test"

In [3]:
DROPOUT = [0.0, 0.2, 0.3, 0.5, 0.7]
SEEDS = [0, 1, 2]
BATCH_SIZE = 256

In [4]:
def create_baseline_model(droput):
    model = BaselineCNN(input_channels = 3, image_size = 32, conv_channels =[64, 128, 256], 
                        kernel_sizes = [3, 3, 3], fc_layers = [128, 64], dropout=droput, num_classes = 10)
    return model

def create_resnet_model(dropout):
    model = torchvision.models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, 10)
    )
    model = model.to(device)
    return model

def create_alexnet_model(dropout):
    model = ModifiedAlexNet(num_classes=10, droput=dropout).to(device)
    return model

def test_dropout_rate(model, train_loader, valid_loader, test_loader, dropouts,
        seeds, scheduler=None, device=None, num_epochs=20):
    
    results = {}
    for dr in dropouts:
        results[dr] = {}
        for seed in seeds:
            print(f"\nDROPOUT={dr} | SEED={seed}")
            set_seed(seed)

            model_instance = model(dr)
            criterion = torch.nn.CrossEntropyLoss()
            optimizer = torch.optim.Adam(model_instance.parameters(), lr=1e-3)

            print("------------------------------------------------------")


            history = train(model_instance, train_loader, valid_loader, criterion, optimizer,
                scheduler=scheduler, device=device, num_epochs=num_epochs, verbose=True, verbose_interval=int(num_epochs/10))
            
            
            validation_accuracy = history['valid_acc'][-1]
            validation_loss = history['valid_loss'][-1]
            test_accuracy, test_loss = evaluate_model(model_instance, test_loader, criterion, device)

            results[dr][seed] = {
                "history": history,
                "valid_acc": validation_accuracy,
                "valid_loss": validation_loss,
                "test_acc": test_accuracy,
                "test_loss": test_loss
            }

            print("------------------------------------------------------")

            print(f"VALIDATION ACCURACY: {validation_accuracy:.4f} | TEST ACCURACY: {test_accuracy:.4f}")

            print("------------------------------------------------------")

    return results

In [5]:
train_loader, valid_loader, test_loader = get_loaders(train_dir=TRAIN_DIR, valid_dir=VALID_DIR,
                                                      test_dir=TEST_DIR, image_size=32, batch_size=BATCH_SIZE)

## Dropout experiments

#### Baseline model

In [6]:
baseline_results = test_dropout_rate(model=create_baseline_model, train_loader=train_loader, valid_loader=valid_loader,
    test_loader=test_loader, dropouts=DROPOUT, seeds=SEEDS, num_epochs=20, device=device)


DROPOUT=0.0 | SEED=0
------------------------------------------------------
Epoch 2/20 | Train Loss: 1.2618 | Valid Loss: 1.4892 | Valid Acc: 0.4800
Epoch 4/20 | Train Loss: 1.0681 | Valid Loss: 1.1893 | Valid Acc: 0.5669
Epoch 6/20 | Train Loss: 0.9440 | Valid Loss: 1.0356 | Valid Acc: 0.6272
Epoch 8/20 | Train Loss: 0.8423 | Valid Loss: 1.4551 | Valid Acc: 0.5399
Epoch 10/20 | Train Loss: 0.7518 | Valid Loss: 1.0098 | Valid Acc: 0.6501
Epoch 12/20 | Train Loss: 0.6691 | Valid Loss: 1.4202 | Valid Acc: 0.5610
Epoch 14/20 | Train Loss: 0.5809 | Valid Loss: 1.1287 | Valid Acc: 0.6315
Epoch 16/20 | Train Loss: 0.5008 | Valid Loss: 1.3598 | Valid Acc: 0.6030
Epoch 18/20 | Train Loss: 0.4279 | Valid Loss: 1.2636 | Valid Acc: 0.6436
Epoch 20/20 | Train Loss: 0.3522 | Valid Loss: 2.1005 | Valid Acc: 0.5476
Best validation accuracy: 0.6501
------------------------------------------------------
VALIDATION ACCURACY: 0.5476 | TEST ACCURACY: 0.5449
-----------------------------------------------

In [7]:
save_results(baseline_results, 'dropout_results_baseline.json')